In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("data/gait-in-neurodegenerative-disease-database-1.0.0")
ts_files = sorted(data_dir.rglob("*.ts"))
print("Number of files:", len(ts_files))
print(ts_files[0])

Number of files: 69
data\gait-in-neurodegenerative-disease-database-1.0.0\.ipynb_checkpoints\als1-checkpoint.ts


In [2]:
columns = [
    "time",
    "left_stride", "right_stride",
    "left_swing", "right_swing",
    "left_swing_pct", "right_swing_pct",
    "left_stance", "right_stance",
    "left_stance_pct", "right_stance_pct",
    "double_support", "double_support_pct"
]



df = pd.read_csv(
    data_dir / "als1.ts",
    sep=r"\s+",
    header=None,
    names=columns
)

display(df.head())

,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
0,22.3200,1.2833,1.3533,0.4067,0.4133,31.69,30.54,0.8767,0.9400,68.31,69.46,0.4633,36.10
1,23.6433,1.3233,1.2667,0.4833,0.4000,36.52,31.58,0.8400,0.8667,63.48,68.42,0.4400,33.25
2,24.9467,1.3033,1.3600,0.4500,0.4267,34.53,31.37,0.8533,0.9333,65.47,68.63,0.4267,32.74
3,26.3633,1.4167,1.2833,0.5033,0.3667,35.53,28.57,0.9133,0.9167,64.47,71.43,0.5467,38.59
4,27.6000,1.2367,1.4067,0.3467,0.3567,28.03,25.36,0.8900,1.0500,71.97,74.64,0.5333,43.13


In [3]:
print(df.shape)
print(df.dtypes)


(194, 13)
time                  float64
left_stride           float64
right_stride          float64
left_swing            float64
right_swing           float64
left_swing_pct        float64
right_swing_pct       float64
left_stance           float64
right_stance          float64
left_stance_pct       float64
right_stance_pct      float64
double_support        float64
double_support_pct    float64
dtype: object


In [4]:
columns = [
    "time",
    "left_stride", "right_stride",
    "left_swing", "right_swing",
    "left_swing_pct", "right_swing_pct",
    "left_stance", "right_stance",
    "left_stance_pct", "right_stance_pct",
    "double_support", "double_support_pct"
]

full_data = {}
als_data = {}
control_data = {}
hunt_data = {}
park_data = {}

for i in range(1,14):
    df = pd.read_csv(
    data_dir / f"als{i}.ts",
    sep=r"\s+",
    header=None,
    names=columns)

    full_data[f"als{i}"] = df
    als_data[f"als{i}"] = df


for i in range(1,17):
    df = pd.read_csv(
    data_dir / f"control{i}.ts",
    sep=r"\s+",
    header=None,
    names=columns)

    full_data[f"control{i}"] = df
    control_data[f"control{i}"] = df


for i in range(1,21):
    df = pd.read_csv(
    data_dir / f"hunt{i}.ts",
    sep=r"\s+",
    header=None,
    names=columns)

    full_data[f"hunt{i}"] = df
    hunt_data[f"hunt{i}"] = df

for i in range(1,16):
    df = pd.read_csv(
    data_dir / f"park{i}.ts",
    sep=r"\s+",
    header=None,
    names=columns)

    full_data[f"park{i}"] = df
    park_data[f"park{i}"] = df


print(len(full_data))

64


In [5]:
percent_cols = ["left_swing_pct", "right_swing_pct", "left_stance_pct", "right_stance_pct", "double_support_pct"]
duration_cols = ["left_stride", "right_stride", "left_swing", "right_swing", "left_stance", "right_stance", "double_support"]


for subject in full_data:
    df = full_data[subject]

    # Check for any missing values
    if (df.isna().sum().sum() != 0):
        print(f"Subject {subject}: Has missing values")

    #Check for any invalid durations (<0)
    if ((df[duration_cols]<=0).any(axis=1).sum() != 0):
        print(f"Subject {subject}: Invalid duration value")

    #Check for any invalid percentages (<0 or >100)
    if ( ((df[percent_cols]<=0) | (df[percent_cols]>=100)).any(axis=1).sum() != 0):
        print(f"Subject {subject}: Invalid percent value")

    #Check for any incorrect calculations
    invalid_right, invalid_left, invalid_double = False, False, False
    for i in range(len(df["time"])):
        if (not invalid_right):
            if (abs(df["right_stride"][i] - df["right_stance"][i] - df["right_swing"][i]) >= 0.01):
                print(f"Subject {subject}: Invalid right stride calculation")
                invalid_right = True

        if (not invalid_left):
            if (abs(df["left_stride"][i] - df["left_stance"][i] - df["left_swing"][i]) >= 0.01):
                print(f"Subject {subject}: Invalid left stride calculation")
                invalid_left = True

        if (not invalid_double):
            if(abs((100*(df["double_support"][i]) / (df["left_swing"][i] + df["left_stance"][i])) - df["double_support_pct"][i]) >= 0.1):
                print(f"Subject {subject}: Invalid double support percent calcuation")
                invalid_double = True

    #Check to make sure time is strictly increasing down the column
    if not (df["time"].diff().dropna() > 0).all():
        print(f"Subject {subject}: Invalid timestamps (nonincreasing)")

    #Check to make sure no columns have just a single value
    constant_cols = df.columns[df.nunique() <= 1]
    if len(constant_cols):
        print(f"Subject {subject}: Constant values (malfunctioning sensor)")

    #Check to make sure swing and stance percentages are computed correctly
    for side in ["left", "right"]:
        for phase in ["swing", "stance"]:
            calculated = (100 * df[f"{side}_{phase}"] / df[f"{side}_stride"])
            error = abs(calculated - df[f"{side}_{phase}_pct"])

            if (error > 0.1).any():
                print(f"Subject {subject}: {side}, {phase} mismatch")

    #Check to make sure no columns have values that remain the same for 5 or more time intervals
    for col in ["left_stride", "right_stride"]:
        runs = df[col].ne(df[col].shift()).cumsum()
        longest_run = df.groupby(runs)[col].size().max()

        if longest_run >= 5:
            print(f"Subject {subject}: {col}, {longest_run}")
        

Subject als5: Invalid percent value
Subject als5: right_stride, 93
Subject als7: Invalid percent value
Subject als12: Invalid percent value
Subject hunt13: Invalid percent value
Subject hunt20: Invalid duration value
Subject hunt20: Invalid percent value
Subject hunt20: Invalid double support percent calcuation
Subject hunt20: right_stride, 59
Subject park7: Invalid percent value
Subject park14: Invalid duration value
Subject park14: Invalid percent value


In [6]:
print(f"Rows greater than 60: {len(full_data["als5"][full_data["als5"]["double_support_pct"]>=60])}")
print(f"Rows with double support greater than 100: {len(full_data["als5"][full_data["als5"]["double_support_pct"]>=100])}")
display(full_data["als5"][full_data["als5"]["double_support_pct"]>=100])


Rows greater than 60: 91
Rows with double support greater than 100: 91


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
114,169.3933,1.5300,1.2533,0.7533,0.4233,49.24,33.78,0.7767,0.83,50.76,66.22,1.5400,100.65
115,178.5767,9.1833,1.2533,0.5233,0.4233,5.70,33.78,8.6600,0.83,94.30,66.22,9.4233,102.61
116,179.9200,1.3433,1.2533,0.4333,0.4233,32.26,33.78,0.9100,0.83,67.74,66.22,1.6733,124.57
117,181.1967,1.2767,1.2533,0.4800,0.4233,37.60,33.78,0.7967,0.83,62.40,66.22,1.5600,122.19
118,182.4700,1.2733,1.2533,0.4600,0.4233,36.13,33.78,0.8133,0.83,63.87,66.22,1.5767,123.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,284.0067,1.1733,1.2533,0.4100,0.4233,34.94,33.78,0.7633,0.83,65.06,66.22,1.5267,130.11
201,285.2233,1.2167,1.2533,0.4333,0.4233,35.62,33.78,0.7833,0.83,64.38,66.22,1.5467,127.12
202,286.4433,1.2200,1.2533,0.4233,0.4233,34.70,33.78,0.7967,0.83,65.30,66.22,1.5600,127.87
203,287.7000,1.2567,1.2533,0.4367,0.4233,34.75,33.78,0.8200,0.83,65.25,66.22,1.5833,125.99


In [7]:
# Looks like sensor malfunctioned at time 169.3933 (row 114), rest of the data is usable. I remove the data past 169s and keep the rest.
full_data["als5"] = full_data["als5"][0:114].reset_index(drop=True)

In [8]:
print(f"Rows greater than 60: {len(full_data["als7"][full_data["als7"]["double_support_pct"]>=60])}")
print(f"Rows with double support greater than 100: {len(full_data["als7"][full_data["als7"]["double_support_pct"]>=100])}")
display(full_data["als7"][full_data["als7"]["double_support_pct"]>=100])

Rows greater than 60: 36
Rows with double support greater than 100: 36


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
1,25.3167,1.7900,1.7600,0.4767,0.7233,26.63,41.10,1.3133,1.0367,73.37,58.90,2.8367,158.47
2,27.1933,1.8767,2.0733,0.5100,0.5500,27.18,26.53,1.3667,1.5233,72.82,73.47,2.7100,144.40
3,29.0900,1.8967,1.9167,0.4667,0.5733,24.60,29.91,1.4300,1.3433,75.40,70.09,2.7700,146.05
4,30.8300,1.7400,1.8933,0.4733,0.5533,27.20,29.23,1.2667,1.3400,72.80,70.77,2.6300,151.15
11,43.4767,1.9233,1.7600,0.4967,0.5633,25.82,32.01,1.4267,1.1967,74.18,67.99,2.4700,128.42
14,49.2933,1.7633,1.9033,0.4333,0.5200,24.57,27.32,1.3300,1.3833,75.43,72.68,2.7267,154.63
15,51.1033,1.8100,1.9433,0.4467,0.5467,24.68,28.13,1.3633,1.3967,75.32,71.87,2.7300,150.83
16,53.0033,1.9000,1.9200,0.4500,0.5533,23.68,28.82,1.4500,1.3667,76.32,71.18,2.8233,148.60
17,54.8100,1.8067,1.9500,0.4633,0.5767,25.65,29.57,1.3433,1.3733,74.35,70.43,2.9167,161.44
35,86.0900,1.8333,1.8100,0.4700,0.5233,25.64,28.91,1.3633,1.2867,74.36,71.09,2.6433,144.18


In [9]:
# Looks like double support sensor malfunctioned sporadically and produced bad measurements occasionally
# Hard to say if any of the double support measurements are valid, but since is data is limited I will only 
# Remove double support measurements that correspond to a bad percent (<100%).

full_data["als7"] = full_data["als7"][full_data["als7"]["double_support_pct"] < 100].reset_index(drop=True)


In [10]:
print(f"Rows with double support greater than 80: {len(full_data["als12"][full_data["als12"]["double_support_pct"]>=80])}")
print(f"Rows with double support greater than 100: {len(full_data["als12"][full_data["als12"]["double_support_pct"]>=100])}")
display(full_data["als12"][full_data["als12"]["double_support_pct"]>=100])


Rows with double support greater than 80: 2
Rows with double support greater than 100: 1


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
110,228.45,1.8133,2.02,0.6767,0.5433,37.32,26.9,1.1367,1.4767,62.68,73.1,3.9633,218.57


In [11]:
# One malfunction at row 110, rest seems valid, I just remove it
full_data["als12"] = full_data["als12"][full_data["als12"]["double_support_pct"] < 100].reset_index(drop=True)

In [12]:
print(f"Rows with double support greater than 80: {len(full_data["hunt13"][full_data["hunt13"]["double_support_pct"]>=80])}")
print(f"Rows with double support greater than 100: {len(full_data["hunt13"][full_data["hunt13"]["double_support_pct"]>=100])}")
display(full_data["hunt13"][full_data["hunt13"]["double_support_pct"]>=100])


Rows with double support greater than 80: 1
Rows with double support greater than 100: 1


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
163,295.3167,1.82,1.4733,0.7933,0.3767,43.59,25.57,1.0267,1.0967,56.41,74.43,1.9533,107.33


In [13]:
# One malfunction at row 163, rest seems valid, I just remove it
full_data["hunt13"] = full_data["hunt13"][full_data["hunt13"]["double_support_pct"] < 100].reset_index(drop=True)

In [14]:
print(f"Rows with double support greater than 80: {len(full_data["park7"][full_data["park7"]["double_support_pct"]>=80])}")
print(f"Rows with double support greater than 100: {len(full_data["park7"][full_data["park7"]["double_support_pct"]>=100])}")
display(full_data["park7"][full_data["park7"]["double_support_pct"]>=100])

Rows with double support greater than 80: 1
Rows with double support greater than 100: 1


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
225,299.9967,2.1767,1.3267,1.1733,0.3367,53.91,25.38,1.0033,0.99,46.09,74.62,2.7233,125.11


In [15]:
# One malfunction at row 225, rest seems valid, I just remove it
full_data["park7"] = full_data["park7"][full_data["park7"]["double_support_pct"] < 100].reset_index(drop=True)

In [16]:
print(f"Rows with double support greater than 80: {len(full_data["park14"][full_data["park14"]["double_support_pct"]>=80])}")
print(f"Rows with double support greater than 100: {len(full_data["park14"][full_data["park14"]["double_support_pct"]>=100])}")
display(full_data["park14"][full_data["park14"]["double_support_pct"]>=100])

Rows with double support greater than 80: 5
Rows with double support greater than 100: 2


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
4,28.4767,1.4200,2.8833,0.4533,2.5067,31.92,86.94,0.9667,0.3767,68.08,13.06,1.4333,100.94
7,32.4333,1.3233,1.2900,0.4267,0.9567,32.24,74.16,0.8967,0.3333,67.76,25.84,1.5933,120.40


In [17]:
# Two malfunctions and a suspicious amount >80, but measurements seem ok, I just remove invalid entries.
full_data["park14"] = full_data["park14"][full_data["park14"]["double_support_pct"] < 100].reset_index(drop=True)

In [18]:
display(full_data["park14"])

,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
0,22.6633,1.5267,1.5400,0.5100,0.7300,33.41,47.40,1.0167,0.8100,66.59,52.60,0.2867,18.78
1,24.1267,1.4633,1.4767,0.5100,0.6733,34.85,45.60,0.9533,0.8033,65.15,54.40,0.2800,19.13
2,25.6033,1.4767,1.5367,0.4633,0.7200,31.38,46.85,1.0133,0.8167,68.62,53.15,1.3900,94.13
3,27.0567,1.4533,1.5367,0.4733,0.7200,32.57,46.85,0.9800,0.8167,67.43,53.15,1.3567,93.35
4,29.7967,1.3200,1.3500,0.4500,0.8833,34.09,65.43,0.8700,0.4667,65.91,34.57,1.2033,91.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...
271,295.8867,0.8933,0.8733,0.2933,0.3233,32.84,37.02,0.6000,0.5500,67.16,62.98,0.2767,30.97
272,296.7933,0.9067,0.9233,0.2767,0.3433,30.51,37.18,0.6300,0.5800,69.49,62.82,0.2867,31.62
273,297.6700,0.8767,0.8700,0.2833,0.3333,32.32,38.31,0.5933,0.5367,67.68,61.69,0.2600,29.66
274,298.5867,0.9167,0.9200,0.2800,0.3633,30.55,39.49,0.6367,0.5567,69.45,60.51,0.2733,29.82


In [19]:
for j in range(len(duration_cols)):
    if len(full_data["park14"][full_data["park14"][duration_cols[j]]<=0]) > 0:
        print(f"{duration_cols[j]} has the invalid duration for park14")

double_support has the invalid duration for park14


In [20]:
print(f"Rows with double support <=0.1: {len(full_data["park14"][full_data["park14"]["double_support"]<=0.1])}")
print(f"Rows with double_support <=0: {len(full_data["park14"][full_data["park14"]["double_support"]<=0])}")
display(full_data["park14"][full_data["park14"]["double_support"]<=0])

Rows with double support <=0.1: 2
Rows with double_support <=0: 2


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
5,31.11,1.3133,1.29,0.4567,0.9567,34.77,74.16,0.8567,0.3333,65.23,25.84,-0.1000,-7.61
6,33.65,1.2167,2.58,0.4367,1.8833,35.89,73.00,0.7800,0.6967,64.11,27.00,-1.1033,-90.68


In [21]:
# I remove the two physically invalid measurements and assume the rest are valid.
full_data["park14"] = full_data["park14"][full_data["park14"]["double_support"]>0].reset_index(drop=True)

In [22]:
for j in range(len(duration_cols)):
    if len(full_data["hunt20"][full_data["hunt20"][duration_cols[j]]<=0]) > 0:
        print(f"{duration_cols[j]} has the invalid duration for hunt20")

double_support has the invalid duration for hunt20


In [23]:
print(f"Rows with double support <=0.1: {len(full_data["hunt20"][full_data["hunt20"]["double_support"]<=0.1])}")
print(f"Rows with double_support <=0: {len(full_data["hunt20"][full_data["hunt20"]["double_support"]<=0])}")
display(full_data["hunt20"][full_data["hunt20"]["double_support"]<=0])

Rows with double support <=0.1: 237
Rows with double_support <=0: 237


,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
0,63.9333,1.2533,42.91,0.3900,0.9833,31.12,2.29,0.8633,41.9267,68.88,97.71,-0.1200,-9.57
1,64.9600,1.0267,42.91,0.3867,0.9833,37.66,2.29,0.6400,41.9267,62.34,97.71,-0.3433,-33.44
2,65.9600,1.0000,42.91,0.3867,0.9833,38.67,2.29,0.6133,41.9267,61.33,97.71,-0.3700,-37.00
3,66.9467,0.9867,42.91,0.3933,0.9833,39.86,2.29,0.5933,41.9267,60.14,97.71,-0.3900,-39.53
4,67.9167,0.9700,42.91,0.3700,0.9833,38.14,2.29,0.6000,41.9267,61.86,97.71,-0.3833,-39.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,294.2733,1.0400,40.09,0.3867,0.8100,37.18,2.02,0.6533,39.2800,62.82,97.98,-0.1567,-15.06
234,296.7233,1.1433,57.09,0.4033,1.3100,35.28,2.29,0.7400,55.7800,64.72,97.71,-0.5700,-49.85
235,297.7200,0.9967,57.09,0.3867,1.3100,38.80,2.29,0.6100,55.7800,61.20,97.71,-0.7000,-70.23
236,298.7167,0.9967,57.09,0.3800,1.3100,38.13,2.29,0.6167,55.7800,61.87,97.71,-0.6933,-69.57


In [24]:
# All of all the double_support measurements are negative (physically impossible of a true measurement), 
# Alongside that the right foot measurements are stagnant for long periods (not plausible)
# I just remove subject hunt20 from the dataset entirely, full sensor malfunction

full_data.pop("hunt20", None)

,time,left_stride,right_stride,left_swing,right_swing,left_swing_pct,right_swing_pct,left_stance,right_stance,left_stance_pct,right_stance_pct,double_support,double_support_pct
0,63.9333,1.2533,42.91,0.3900,0.9833,31.12,2.29,0.8633,41.9267,68.88,97.71,-0.1200,-9.57
1,64.9600,1.0267,42.91,0.3867,0.9833,37.66,2.29,0.6400,41.9267,62.34,97.71,-0.3433,-33.44
2,65.9600,1.0000,42.91,0.3867,0.9833,38.67,2.29,0.6133,41.9267,61.33,97.71,-0.3700,-37.00
3,66.9467,0.9867,42.91,0.3933,0.9833,39.86,2.29,0.5933,41.9267,60.14,97.71,-0.3900,-39.53
4,67.9167,0.9700,42.91,0.3700,0.9833,38.14,2.29,0.6000,41.9267,61.86,97.71,-0.3833,-39.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,295.5800,1.3067,40.09,0.6733,0.8100,51.53,2.02,0.6333,39.2800,48.47,97.98,56.4133,4317.35
234,296.7233,1.1433,57.09,0.4033,1.3100,35.28,2.29,0.7400,55.7800,64.72,97.71,-0.5700,-49.85
235,297.7200,0.9967,57.09,0.3867,1.3100,38.80,2.29,0.6100,55.7800,61.20,97.71,-0.7000,-70.23
236,298.7167,0.9967,57.09,0.3800,1.3100,38.13,2.29,0.6167,55.7800,61.87,97.71,-0.6933,-69.57


In [25]:

for subject in full_data:
    df = full_data[subject]

    # Check for any missing values
    if (df.isna().sum().sum() != 0):
        print(f"Subject {subject}: Has missing values")

    #Check for any invalid durations (<0)
    if ((df[duration_cols]<=0).any(axis=1).sum() != 0):
        print(f"Subject {subject}: Invalid duration value")

    #Check for any invalid percentages (<0 or >100)
    if ( ((df[percent_cols]<=0) | (df[percent_cols]>=100)).any(axis=1).sum() != 0):
        print(f"Subject {subject}: Invalid percent value")

    #Check for any incorrect calculations
    invalid_right, invalid_left, invalid_double = False, False, False
    for i in range(len(df["time"])):
        if (not invalid_right):
            if (abs(df["right_stride"][i] - df["right_stance"][i] - df["right_swing"][i]) >= 0.01):
                print(f"Subject {subject}: Invalid right stride calculation")
                invalid_right = True

        if (not invalid_left):
            if (abs(df["left_stride"][i] - df["left_stance"][i] - df["left_swing"][i]) >= 0.01):
                print(f"Subject {subject}: Invalid left stride calculation")
                invalid_left = True

        if (not invalid_double):
            if(abs((100*(df["double_support"][i]) / (df["left_swing"][i] + df["left_stance"][i])) - df["double_support_pct"][i]) >= 0.1):
                print(f"Subject {subject}: Invalid double support percent calcuation")
                invalid_double = True

    #Check to make sure time is strictly increasing down the column
    if not (df["time"].diff().dropna() > 0).all():
        print(f"Subject {subject}: Invalid timestamps (nonincreasing)")

    #Check to make sure no columns have just a single value
    constant_cols = df.columns[df.nunique() <= 1]
    if len(constant_cols):
        print(f"Subject {subject}: Constant values (malfunctioning sensor)")

    #Check to make sure swing and stance percentages are computed correctly
    for side in ["left", "right"]:
        for phase in ["swing", "stance"]:
            calculated = (100 * df[f"{side}_{phase}"] / df[f"{side}_stride"])
            error = abs(calculated - df[f"{side}_{phase}_pct"])

            if (error > 0.1).any():
                print(f"Subject {subject}: {side}, {phase} mismatch")

    #Check to make sure no columns have values that remain the same for 5 or more time intervals
    for col in ["left_stride", "right_stride"]:
        runs = df[col].ne(df[col].shift()).cumsum()
        longest_run = df.groupby(runs)[col].size().max()

        if longest_run >= 5:
            print(f"Subject {subject}: {col}, {longest_run}")

# Nothing prints, thus all measurements are physically valid

In [26]:
# combined = pd.concat(
#     [df.assign(subject=subject) for subject, df in full_data.items()],
#     ignore_index=True
# )

# combined.to_csv("full_data_clean.csv", index=False)